# Weekend Put-Sell Strategy Backtest

**Strategy:** Sell ATM puts on Thursday (last Israeli trading day) near session close,
buy them back on Sunday (first trading day of the following week) near session open.

**Universe:** TA-35 monthly put options, 2014-2023.

**Pricing:** Best bid/ask midpoint from quote snapshots.

| Parameter | Value |
|---|---|
| Entry window | Thursday >= 17:10 (last ~15 min) |
| Exit window | Sunday <= 09:45 (first ~15 min) |
| Strike | ATM (closest to median spot) |
| Expiry | Nearest monthly, >= 3 days out |

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DUCKDB_PATH = str(Path.home() / "fg3-data" / "maof.duckdb")

# --- Configurable parameters ---
ENTRY_TIME_MIN = "17100000"  # Thursday: quotes from 17:10:00.00 onward
EXIT_TIME_MAX  = "09450000"  # Sunday: quotes until 09:45:00.00
MIN_DAYS_TO_EXPIRY = 3       # skip options expiring within 3 days

con = duckdb.connect(DUCKDB_PATH, read_only=True)
print(f"Connected to {DUCKDB_PATH}")
con.execute("SHOW TABLES").fetchdf()

Connected to /Users/yschiff/fg3-data/maof.duckdb


,name
0,_metadata
1,quotes
2,trades


## Load Data

Pull all **put** quotes on **Thursdays** (dow=4) and **Sundays** (dow=0) with valid
two-sided markets (both bid and ask > 0).

In [ ]:
query = """
SELECT
    date,
    timestamp,
    strike,
    expiry_date,
    bid_price_1,
    ask_price_1,
    underlying_index,
    (bid_price_1 + ask_price_1) / 2.0 AS midpoint
FROM quotes
WHERE option_type = 'P'
  AND bid_price_1 > 0
  AND ask_price_1 > 0
  AND underlying_index IS NOT NULL
  AND underlying_index > 0
  AND (extract(dow FROM date) = 4 OR extract(dow FROM date) = 0)
ORDER BY date, timestamp, strike
"""

df = con.execute(query).fetchdf()
df["date"] = pd.to_datetime(df["date"])
df["expiry_date"] = pd.to_datetime(df["expiry_date"])

print(f"Loaded {len(df):,} put quotes on Thursdays and Sundays")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique dates: {df['date'].nunique()}")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Error: KeyboardInterrupt: <EMPTY MESSAGE>

At:
  /Users/yschiff/.fg3-venv/lib/python3.14/site-packages/traitlets/traitlets.py(708): __set__
  /var/folders/g6/4b91ml114pq677yqv52qb9080000gn/T/ipykernel_89937/1472433507.py(21): <module>
  /Users/yschiff/.fg3-venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py(3747): run_code
  /Users/yschiff/.fg3-venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py(3687): run_ast_nodes
  /Users/yschiff/.fg3-venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py(3446): run_cell_async
  /Users/yschiff/.fg3-venv/lib/python3.14/site-packages/IPython/core/async_helpers.py(128): _pseudo_sync_runner
  /Users/yschiff/.fg3-venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py(3224): _run_cell
  /Users/yschiff/.fg3-venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py(3169): run_cell
  /Users/yschiff/.fg3-venv/lib/python3.14/site-packages/ipykernel/zmqshell.py(663): run_cell
  /Users/yschiff/.fg3-v

## Trading Calendar

Build Thursday-Sunday pairs from actual trading dates.
Skip weeks where either day is missing (holidays).

In [ ]:
trading_dates = df["date"].drop_duplicates().sort_values().reset_index(drop=True)
trading_dates_set = set(trading_dates)

# pandas: Monday=0 … Sunday=6 → Thursday=3, Sunday=6
thursdays = trading_dates[trading_dates.dt.dayofweek == 3].values

pairs = []
for thu in thursdays:
    sun = thu + np.timedelta64(3, "D")
    if sun in trading_dates_set:
        pairs.append((pd.Timestamp(thu), pd.Timestamp(sun)))

pairs_df = pd.DataFrame(pairs, columns=["entry_date", "exit_date"])
print(f"Found {len(pairs_df)} valid Thursday → Sunday pairs")
if len(pairs_df) > 0:
    print(f"First: {pairs_df.iloc[0]['entry_date'].date()} → {pairs_df.iloc[0]['exit_date'].date()}")
    print(f"Last:  {pairs_df.iloc[-1]['entry_date'].date()} → {pairs_df.iloc[-1]['exit_date'].date()}")

## Strategy Engine

For each Thursday-Sunday pair:
1. **Thursday near-close:** find the median spot, pick ATM strike in the nearest valid expiry, record the last-quote midpoint as the sell price.
2. **Sunday near-open:** find the same contract, record the first-quote midpoint as the buy price.
3. **P&L** = sell − buy.

In [ ]:
entry_dates = set(pairs_df["entry_date"])
exit_dates = set(pairs_df["exit_date"])

thu_late = df[(df["date"].isin(entry_dates)) & (df["timestamp"] >= ENTRY_TIME_MIN)]
sun_early = df[(df["date"].isin(exit_dates)) & (df["timestamp"] <= EXIT_TIME_MAX)]

thu_by_date = dict(list(thu_late.groupby("date")))
sun_by_date = dict(list(sun_early.groupby("date")))

trades_list = []
skipped = {"no_thu_quotes": 0, "no_valid_expiry": 0, "no_sun_quotes": 0}

for _, pair in pairs_df.iterrows():
    thu_date, sun_date = pair["entry_date"], pair["exit_date"]

    # --- Thursday: late-session put quotes ---
    thu_q = thu_by_date.get(thu_date)
    if thu_q is None or thu_q.empty:
        skipped["no_thu_quotes"] += 1
        continue

    spot = thu_q["underlying_index"].median()

    # Nearest monthly expiry at least MIN_DAYS_TO_EXPIRY away
    min_expiry = thu_date + pd.Timedelta(days=MIN_DAYS_TO_EXPIRY)
    valid_expiries = thu_q.loc[thu_q["expiry_date"] >= min_expiry, "expiry_date"].unique()
    if len(valid_expiries) == 0:
        skipped["no_valid_expiry"] += 1
        continue
    target_expiry = valid_expiries.min()

    # ATM strike (closest to spot) within the target expiry
    thu_exp = thu_q[thu_q["expiry_date"] == target_expiry]
    strikes = thu_exp["strike"].unique()
    atm_strike = int(strikes[np.argmin(np.abs(strikes - spot))])

    # Sell at the last available quote's midpoint
    atm_thu = thu_exp[thu_exp["strike"] == atm_strike].sort_values("timestamp")
    sell_price = atm_thu.iloc[-1]["midpoint"]

    # --- Sunday: early-session quote for the same contract ---
    sun_q = sun_by_date.get(sun_date)
    if sun_q is None or sun_q.empty:
        skipped["no_sun_quotes"] += 1
        continue

    sun_contract = sun_q[
        (sun_q["strike"] == atm_strike) & (sun_q["expiry_date"] == target_expiry)
    ]
    if sun_contract.empty:
        skipped["no_sun_quotes"] += 1
        continue

    # Buy back at the first available quote's midpoint
    buy_price = sun_contract.sort_values("timestamp").iloc[0]["midpoint"]

    trades_list.append({
        "entry_date": thu_date,
        "exit_date": sun_date,
        "strike": atm_strike,
        "expiry": target_expiry,
        "spot": round(spot, 2),
        "sell_price": sell_price,
        "buy_price": buy_price,
        "pnl": sell_price - buy_price,
    })

results = pd.DataFrame(trades_list)
print(f"Executed {len(results)} trades")
print(f"Skipped: {skipped}")

## Results

In [ ]:
if not results.empty:
    show = results.copy()
    for col in ["entry_date", "exit_date", "expiry"]:
        show[col] = pd.to_datetime(show[col]).dt.date
    show[["sell_price", "buy_price", "pnl"]] = show[["sell_price", "buy_price", "pnl"]].round(1)

    print(f"Total trades: {len(show)}\n")
    print("First 10 trades:")
    display(show.head(10))
    print("\nLast 10 trades:")
    display(show.tail(10))
else:
    print("No trades executed.")

## Summary Statistics

In [ ]:
if not results.empty:
    pnl = results["pnl"]
    cum = pnl.cumsum()
    drawdown = cum - cum.cummax()
    wins = (pnl > 0).sum()

    # Longest losing streak
    is_loss = (pnl <= 0).astype(int)
    groups = (is_loss != is_loss.shift()).cumsum()
    streaks = is_loss * (is_loss.groupby(groups).cumcount() + 1)

    sharpe = pnl.mean() / pnl.std() * np.sqrt(52) if pnl.std() > 0 else np.nan

    stats = {
        "Total Trades": len(pnl),
        "Total P&L (ILS)": f"{pnl.sum():,.1f}",
        "Mean P&L / Trade": f"{pnl.mean():.2f}",
        "Median P&L / Trade": f"{pnl.median():.2f}",
        "Std Dev": f"{pnl.std():.2f}",
        "Win Rate": f"{wins}/{len(pnl)} ({100 * wins / len(pnl):.1f}%)",
        "Best Trade": f"{pnl.max():.1f}",
        "Worst Trade": f"{pnl.min():.1f}",
        "Sharpe (annualized)": f"{sharpe:.2f}",
        "Max Drawdown (ILS)": f"{drawdown.min():.1f}",
        "Longest Losing Streak": int(streaks.max()),
    }

    for k, v in stats.items():
        print(f"{k:>25s}:  {v}")

## Charts

In [ ]:
if not results.empty:
    fig, ax = plt.subplots(figsize=(14, 5))
    dates = pd.to_datetime(results["entry_date"])
    cum_pnl = results["pnl"].cumsum()

    ax.plot(dates, cum_pnl, linewidth=1.2, color="steelblue")
    ax.fill_between(dates, cum_pnl, alpha=0.15, color="steelblue")
    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
    ax.set_title("Weekend Put-Sell: Cumulative P&L", fontsize=14)
    ax.set_xlabel("Date")
    ax.set_ylabel("Cumulative P&L (ILS / contract)")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

In [ ]:
if not results.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    pnl = results["pnl"]

    ax.hist(pnl, bins=50, edgecolor="white", alpha=0.7, color="steelblue")
    ax.axvline(pnl.mean(), color="red", linestyle="--", lw=1.2,
               label=f"Mean: {pnl.mean():.1f}")
    ax.axvline(pnl.median(), color="orange", linestyle="--", lw=1.2,
               label=f"Median: {pnl.median():.1f}")
    ax.axvline(0, color="grey", linewidth=0.8)
    ax.set_title("P&L Distribution per Trade", fontsize=14)
    ax.set_xlabel("P&L (ILS / contract)")
    ax.set_ylabel("Frequency")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

In [ ]:
if not results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # --- Annual P&L bar chart ---
    results["year"] = pd.to_datetime(results["entry_date"]).dt.year
    annual = results.groupby("year")["pnl"].agg(["sum", "count", "mean"])
    colors = ["forestgreen" if v >= 0 else "indianred" for v in annual["sum"]]
    axes[0].bar(annual.index, annual["sum"], color=colors, edgecolor="white")
    axes[0].axhline(0, color="grey", linewidth=0.5)
    axes[0].set_title("Annual P&L", fontsize=13)
    axes[0].set_xlabel("Year")
    axes[0].set_ylabel("Total P&L (ILS)")
    axes[0].grid(True, alpha=0.3, axis="y")

    # --- Drawdown chart ---
    dates = pd.to_datetime(results["entry_date"])
    cum_pnl = results["pnl"].cumsum()
    dd = cum_pnl - cum_pnl.cummax()
    axes[1].fill_between(dates, dd, color="indianred", alpha=0.4)
    axes[1].plot(dates, dd, color="indianred", linewidth=0.8)
    axes[1].set_title("Drawdown", fontsize=13)
    axes[1].set_xlabel("Date")
    axes[1].set_ylabel("Drawdown (ILS)")
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

    # Annual summary table
    annual.columns = ["Total P&L", "Trades", "Avg P&L"]
    display(annual.round(1))